## 01_xi_cc — Coles 自己相関関数 ξ_CC(r)

**入力**
- `output/data/coles_locations.csv`  — Coles 店舗座標 (N_D = 685)
- `output/data/random_masked.csv`    — 共有大陸マスク済みランダムカタログ (N_R = 10,390)

**出力**
- `output/xi/xi_cc.csv`             — ξ_CC(r): r_km, xi, xi_err, n_rr (38 ビン)

**前提**
- `00_random_catalog.ipynb` 実行済み（`random_masked.csv` が存在すること）
- `gradle :lib:jar` 完了

**解析パラメータ**
- 距離: Haversine [km] / ビン: 対数等分, Δln r ≈ 0.15, rMin = meanNN/2, rMax = 2000 km
- 推定量: Landy-Szalay  ξ̂(r) = (DD − 2DR + RR) / RR

In [1]:
@file:DependsOn("../../lib/build/libs/retail-utils-1.0.jar")

In [2]:
%use dataframe
%use lets-plot

import retail.*
import kotlin.math.*

In [3]:
// --- データ読み込み ---
val dfColes  = DataFrame.readCSV("./output/data/coles_locations.csv")
val dfRandom = DataFrame.readCSV("./output/data/random_masked.csv")

val dataPoints: List<Point> = dfColes.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }
val randomPoints: List<Point> = dfRandom.rows().map { Point(it["lat"] as Double, it["lon"] as Double) }

val nD = dataPoints.size
val nR = randomPoints.size
println("N_D (Coles)  = $nD")
println("N_R (random) = $nR")

N_D (Coles)  = 685
N_R (random) = 10390


In [4]:
// --- ビン設計: rMin = meanNN / 2, Δln r ≈ 0.15 ---
val nnDistances = dataPoints.map { p1 ->
    dataPoints.filter { it !== p1 }.minOf { p2 -> haversine(p1, p2) }
}
val meanNN = nnDistances.average()

val bins     = logBins(rMin = meanNN / 2.0)
val centers  = binCenters(bins)
val nBins    = centers.size

println("meanNN = %.2f km".format(meanNN))
println("rMin   = %.2f km  (= meanNN / 2)".format(meanNN / 2.0))
println("nBins  = $nBins  (Δln r = %.3f)".format(ln(2000.0 / (meanNN / 2.0)) / nBins))

meanNN = 15.25 km
rMin   = 7.63 km  (= meanNN / 2)
nBins  = 38  (Δln r = 0.147)


In [5]:
// --- DD / DR / RR ペアカウント ---
println("DD を計算中...")
val nDD = pairCounts(dataPoints, null, bins)

println("DR を計算中...")
val nDR = pairCounts(dataPoints, randomPoints, bins)

println("RR を計算中...  (N_R^2/2 ≈ ${"%,.0f".format(nR.toLong() * (nR - 1) / 2.0)} ペア)")
val nRR = pairCounts(randomPoints, null, bins)

println("完了")

DD を計算中...
DR を計算中...
RR を計算中...  (N_R^2/2 ≈ 53,970,855 ペア)
完了


In [6]:
// --- Landy-Szalay 推定量 ---
val normDD = nD.toLong() * (nD - 1) / 2
val normDR = nD.toLong() * nR
val normRR = nR.toLong() * (nR - 1) / 2

val xiCC: List<XiBin> = landySzalay(nDD, nDR, nRR, normDD, normDR, normRR, bins)

println("%-8s  %-9s  %-9s  %s".format("r [km]", "ξ_CC", "σ_ξ", "S/N"))
println("-".repeat(40))
xiCC.forEach { b ->
    val sn = if (b.xiErr > 0 && !b.xiErr.isNaN()) b.xi / b.xiErr else Double.NaN
    println("%-8.1f  %-9.4f  %-9.5f  %.1f".format(b.rCenter, b.xi, b.xiErr, sn))
}

r [km]    ξ_CC       σ_ξ        S/N
----------------------------------------
8.2       232.3009   10.80746   21.5
9.5       251.8541   10.49920   24.0
11.0      217.6717   7.76037    28.0
12.7      187.0023   5.65564    33.1
14.8      181.6368   4.85009    37.5
17.1      164.9913   3.86444    42.7
19.8      142.4958   2.88146    49.5
22.9      120.7910   2.11022    57.2
26.5      100.0610   1.51804    65.9
30.7      76.6338    1.00334    76.4
35.5      55.8458    0.63158    88.4
41.1      42.4625    0.42117    100.8
47.6      33.4836    0.29029    115.3
55.2      24.4917    0.18692    131.0
63.9      17.0336    0.11372    149.8
73.9      11.1122    0.06614    168.0
85.6      7.7701     0.04160    186.8
99.1      5.2845     0.02577    205.0
114.8     5.2873     0.02245    235.5
132.9     3.6331     0.01435    253.3
153.9     2.7776     0.01017    273.2
178.2     1.5838     0.00606    261.5
206.3     1.2071     0.00450    268.2
238.8     1.8593     0.00509    365.4
276.5     0.7128     0

In [7]:
// --- ξ_CC(r) プロット (log-x) ---
val rVec    = xiCC.map { it.rCenter }
val xiVec   = xiCC.map { it.xi }
val xiLo    = xiCC.map { it.xi - it.xiErr }
val xiHi    = xiCC.map { it.xi + it.xiErr }

letsPlot(mapOf("r" to rVec, "xi" to xiVec, "lo" to xiLo, "hi" to xiHi)) +
    geomRibbon(alpha = 0.20, fill = "#4682B4") { x = "r"; ymin = "lo"; ymax = "hi" } +
    geomLine(color = "#4682B4", size = 1.3) { x = "r"; y = "xi" } +
    geomPoint(color = "#4682B4", size = 2.0) { x = "r"; y = "xi" } +
    geomHLine(yintercept = 0.0, linetype = "dashed", color = "#808080") +
    geomVLine(xintercept = 71.3,  linetype = "dotted", color = "#666666") +
    geomVLine(xintercept = 666.0, linetype = "dotted", color = "#CC4444") +
    scaleXLog10(name = "分離距離 r [km]") +
    scaleYContinuous(name = "ξ_CC(r)") +
    ggtitle("Coles 自己相関関数 ξ_CC(r)",
            "点線: r_D = 71.3 km (Debye Length) | 赤点線: r_BAO = 666 km") +
    ggsize(800, 450)

<path d="M0.0 39.42631638275765 L0.0 39.42631638275765 L19.663251731699745 15.999999999999943 L39.32650346339949 60.94307012681526 L58.98975519509918 100.83803358918519 L78.65300692679892 108.34989292940494 L98.31625865849873 129.81173030504962 L117.97951039019841 158.3913619199937 L137.64276212189816 185.7506520406952 L157.30601385359785 211.70553583592448 L176.9692655852976 240.84920636323199 L196.63251731699734 266.6063646432751 L216.29576904869708 283.15350494017474 L235.95902078039677 294.24261945649914 L255.62227251209657 305.31393987009534 L275.28552424379626 314.4816278035457 L294.94877597549595 321.74744461160964 L314.61202770719564 325.8455606090063 L334.27527943889544 328.8904827107181 L353.9385311705951 328.8911702131724 L373.60178290229493 330.9145398693519 L393.2650346339946 331.9610188586912 L412.9282863656944 333.4192360862192 L432.591538097394 333.8797193263452 L452.2547898290939 333.0851049435867 L471.9180415607935 334.4836174880088 L491.5812932924933 334.81837246677037 L511.2445450241931 334.6614219215017 L530.9077967558926 334.5803394729539 L550.5710484875924 334.86115251220104 L570.2343002192922 334.5151834336363 L589.897551950992 331.42938812958073 L609.5608036826916 333.55318934305086 L629.2240554143914 335.0654511019072 L648.8873071460912 335.3648704694032 L668.5505588777908 335.1045182037519 L688.2138106094906 334.73028691095595 L707.8770623411904 335.56345402982623 L727.5403140728902 335.9995188309035 L727.5403140728902 336.0 L707.8770623411904 335.56431850828767 L688.2138106094906 334.73193800722885 L668.5505588777908 335.10592544913015 L648.8873071460912 335.3661316139458 L629.2240554143914 335.06718688301845 L609.5608036826916 333.5570587733086 L589.897551950992 331.4367876439415 L570.2343002192922 334.5185221629505 L550.5710484875924 334.86429833701584 L530.9077967558926 334.58450009983267 L511.2445450241931 334.6659667679131 L491.5812932924933 334.82313091046893 L471.9180415607935 334.49010303633 L452.2547898290939 333.09749321450323 L432.591538097394 333.89067780357925 L412.9282863656944 333.43397907611245 L393.2650346339946 331.9857722035956 L373.60178290229493 330.9494650783181 L353.9385311705951 328.9458294922441 L334.27527943889544 328.9532286104077 L314.61202770719564 325.9468317124109 L294.94877597549595 321.90846629756953 L275.28552424379626 314.75849022538966 L255.62227251209657 305.7690025548088 L235.95902078039677 294.949343737343 L216.29576904869708 284.17886674685883 L196.63251731699734 268.14397209988573 L176.9692655852976 243.29186551072002 L157.30601385359785 215.40126472245566 L137.64276212189816 190.88806480427795 L117.97951039019841 165.40639426825302 L98.31625865849873 139.2198647094331 L78.65300692679892 120.1576262770676 L58.98975519509918 114.60690843661254 L39.32650346339949 79.8359792928986 L19.663251731699745 41.56068711239874 L0.0 65.73747080749018 Z" fill="rgb(70,130,180)" stroke-width="1.0" fill-opacity="0.2">
 
 
 
 <path d="M0.0 39.42631638275765 L0.0 39.42631638275765 L19.663251731699745 15.999999999999943 L39.32650346339949 60.94307012681526 L58.98975519509918 100.83803358918519 L78.65300692679892 108.34989292940494 L98.31625865849873 129.81173030504962 L117.97951039019841 158.3913619199937 L137.64276212189816 185.7506520406952 L157.30601385359785 211.70553583592448 L176.9692655852976 240.84920636323199 L196.63251731699734 266.6063646432751 L216.29576904869708 283.15350494017474 L235.95902078039677 294.24261945649914 L255.62227251209657 305.31393987009534 L275.28552424379626 314.4816278035457 L294.94877597549595 321.74744461160964 L314.61202770719564 325.8455606090063 L334.27527943889544 328.8904827107181 L353.9385311705951 328.8911702131724 L373.60178290229493 330.9145398693519 L393.2650346339946 331.9610188586912 L412.9282863656944 333.4192360862192 L432.591538097394 333.8797193263452 L452.2547898290939 333.0851049435867 L471.9180415607935 334.4836174880088 L491.5812932924933 334.81837246677037 L511.2445450241931 334.6614219215017 L530.9077967558926 334.58033947

In [8]:
// --- xi_cc.csv 出力 ---
val outPath = "./output/xi/xi_cc.csv"
val outFile = java.io.File(outPath)
outFile.bufferedWriter().use { w ->
    w.appendLine("r_km,xi,xi_err,n_rr")
    xiCC.forEach { b -> w.appendLine("${b.rCenter},${b.xi},${b.xiErr},${b.nRR}") }
}
println("保存: $outPath  ($nBins 行)")

// BAO ビンのサマリー
val baoIdx = xiCC.indices.minByOrNull { abs(xiCC[it].rCenter - 666.0) }!!
val bao = xiCC[baoIdx]
println()
println("=== Retail BAO サマリー (r ≈ ${bao.rCenter.toInt()} km) ===")
println("  ξ_CC      = %.4f".format(bao.xi))
println("  σ_Poisson = %.5f".format(bao.xiErr))
println("  S/N_P     = %.1f σ".format(bao.xi / bao.xiErr))

保存: ./output/xi/xi_cc.csv  (38 行)

=== Retail BAO サマリー (r ≈ 666 km) ===
  ξ_CC      = 3.2215
  σ_Poisson = 0.00304
  S/N_P     = 1059.9 σ
